[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KFUPM-JRCAI/star-instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection.ipynb)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Building the prompts dataset

In [2]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')

  0%|          | 0/10 [00:00<?, ?it/s]

In [3]:
len(prompts)

245

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [4]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED', prompts))
len(filtered_prompts)

117

## Baseline with emotone_ar dataset

### Get the dataset prompts

In [5]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: 'emotone_ar' in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(filtered_prompts)

117

### Download the dataset

In [6]:
import datasets

In [7]:
emotone_ar_experimental = datasets.load_dataset('MagedSaeed/emotone_ar_experimental')
emotone_ar_experimental

DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 8052
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 2013
    })
})

### Merge the prompts

In [8]:
from jinja2 import Environment, StrictUndefined

In [9]:
def apply_template(prompt_template, sample):
    template = prompt_template['template']
    sample['answer_choices'] = prompt_template['answer_choices']
    env = Environment(undefined=StrictUndefined)
    if "|||" not in template:
        raise ValueError("No ||| dividor")
    template = env.from_string(template)
    rendered_template = template.render(**sample)
    return rendered_template

see how the template is applied on different examples

### Perform generation on one example prompt, for experimentation

In [10]:
example_prompt_template = dataset_prompts[4]
print(apply_template(example_prompt_template, emotone_ar_experimental['train'][2]))

Review this tweet المشكله ليست فيمن يخذلك ، يخونك ، يوجعك ، يسحقك ، المشكله هي انك تتمكن من تصديق شخص نال منك مره لتمنحه فرصه النيل منك اخري ! carefully and then identify the emotion being expressed. Choose from the following options:none or anger or joy or sadness or love or sympathy or surprise or fear.
|||
sadness


In [11]:
rendered_test_prompts_dataset = list(map(lambda sample: apply_template(example_prompt_template, sample), emotone_ar_experimental['test']))
len(rendered_test_prompts_dataset)

2013

## Evaluate the LLM

In [12]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [19]:
import torch
# MODEL_PATH = '/raid_storage/shared_models/Meta-Llama-3.1-8B-Instruct'
# MODEL_PATH = '/raid_storage/shared_models/Meta-Llama-3.1-8B'
# MODEL_PATH = '/hdd/shared_models/jais-13b-chat'
MODEL_PATH = '/hdd/shared_models/jais-13b'
TUNED_MODEL_PATH = './tuned_models/jais-13b'
TOKENIZER_PATH = MODEL_PATH

In [14]:
def load_model_and_tokenizer(model_name=MODEL_PATH, tokenizer_name=TOKENIZER_PATH):
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, trust_remote_code=True)
    return model, tokenizer

In [15]:
model,tokenizer = load_model_and_tokenizer()

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
from peft import PeftModel
if TUNED_MODEL_PATH:
    print('loading tuned model')
    model = PeftModel.from_pretrained(model,TUNED_MODEL_PATH)

loading tuned model weights


/home/majed_alshaibani/Projects/InstructionsTuning/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:1150: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [21]:
# check tokenizer spaces
leftspace_counter = 0
rightspace_counter = 0
for vocab,vocab_id in tqdm(tokenizer.get_vocab().items()):
    if vocab.startswith('Ġ'):
        leftspace_counter += 1
    elif vocab.endswith('Ġ'):
        rightspace_counter += 1
leftspace_counter,rightspace_counter

  0%|          | 0/84992 [00:00<?, ?it/s]

(59030, 98)

In [27]:
def compute_probability(model, tokenizer, prompt, option):
    prompt = prompt.strip()
    # check if the model is sensitive to preprocessing! Better to pay attention to these details.
    prompt = prompt.strip('.')
    # prompt = prompt+":"
    prefix_inputs = tokenizer(prompt, add_special_tokens=True, return_tensors='pt').to('cuda')
    suffix = f' {option}' # very important to add a space here as the tokenizer favors adding spaces at the begining of the word, NOT at the last.
    suffix_inputs = tokenizer(suffix, add_special_tokens=False, return_tensors='pt').to('cuda')
    model.eval()
    with torch.no_grad():
        # generate only one token, return its probability as the option probability
        output_token_id = suffix_inputs['input_ids'][:,0]
        outputs = model(**prefix_inputs)
        next_token_logits = outputs.logits.log_softmax(dim=-1)[0,-1] # this should give a shape of [vocab_size]
        # assert next_token_logits.shape[0] == tokenizer.vocab_size, f"Expected {tokenizer.vocab_size} but got {next_token_logits.shape[0]}"
        next_token_probs = next_token_logits[output_token_id]
    return next_token_probs.item()

In [28]:
example_prompt_template['answer_choices']

['none', 'anger', 'joy', 'sadness', 'love', 'sympathy', 'surprise', 'fear']

In [29]:
def evaluate_llm(model, tokenizer, dataset):
    # dataset = dataset[:100]
    correct_predictions = 0
    for prompt in tqdm(dataset):
        prompt, expected_output = prompt.split("|||")
        prompt = prompt.strip()
        expected_output = expected_output.strip()
        probabilities = []
        options = example_prompt_template['answer_choices']
        for option in options:
            prob = compute_probability(model, tokenizer, prompt, option)
            if prob is None:
                continue
            probabilities.append(prob)
        print('probs:',probabilities)
        predicted_index = probabilities.index(max(probabilities))
        predicted_output = options[predicted_index].strip()
        print('predicted output:',predicted_output, 'expected output:',expected_output)
        print('-'*80)
        if predicted_output == expected_output:
            correct_predictions += 1
    accuracy = correct_predictions / len(dataset)
    return accuracy

In [30]:
accuracy = evaluate_llm(model, tokenizer, dataset=rendered_test_prompts_dataset)
print(f"Accuracy: {accuracy:.4f}")

  0%|          | 0/2013 [00:00<?, ?it/s]

probs: [-3.998365640640259, -0.701887309551239, -1.2228147983551025, -2.6552507877349854, -2.6004068851470947, -5.1310882568359375, -3.635965585708618, -6.128679275512695]
predicted output: anger expected output: anger
--------------------------------------------------------------------------------
probs: [-7.070573806762695, -7.593660354614258, -5.905361175537109, -2.2776222229003906, -3.9698104858398438, -0.1418781876564026, -6.930318832397461, -9.724099159240723]
predicted output: sympathy expected output: sympathy
--------------------------------------------------------------------------------
probs: [-6.037508010864258, -4.178936004638672, -8.048588752746582, -5.4101104736328125, -9.933201789855957, -7.1072540283203125, -3.9781837463378906, -0.047235555946826935]
predicted output: fear expected output: fear
--------------------------------------------------------------------------------
probs: [-1.5714298486709595, -1.0904918909072876, -6.352261066436768, -4.503742694854736, -8.37